In [ ]:
#!pip uninstall huetracer -y
#!pip install huetracer
import scanpy as sc
import matplotlib.pyplot as plt
from matplotlib_inline.backend_inline import set_matplotlib_formats
import seaborn as sns
import os
import random
import numpy as np
import pandas as pd
import scvi
import gc
import math
import bin2cell as b2c
import torch
from itertools import cycle
from sklearn.neighbors import NearestNeighbors
import importlib
import gdown
import zipfile
import adjustText as at
import scipy
import huetracer
import warnings

pd.set_option('display.max_columns', None)
sc.set_figure_params(figsize=[10,10],dpi=100)
sns.set_style("whitegrid")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

scvi.settings.seed = 0
SEED = 42

np.random.seed(SEED)
random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
device_str = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print(f"Using device: {device}")
warnings.filterwarnings("ignore", message=".*non-integers were found.*")

In [ ]:
### parameters to be input
SAMPLE_NAME = 'E16_15'
lib_id = SAMPLE_NAME # list(sp_adata.uns['spatial'].keys())[0]

path = os.path.expanduser("~")+"/Desktop/space/" + SAMPLE_NAME
save_path_for_today = os.path.expanduser("~")+"/tmp/outputs/250610_" + SAMPLE_NAME
visium_path = "/Volumes/Public/data/tendon_mouse/visium_HD/"
source_image_path = visium_path + "he/" + SAMPLE_NAME + ".tif"
expression_path = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_002um"
### optional 8/16 um binned dataset
expression_path_8um = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_008um"
expression_path_16um = visium_path + SAMPLE_NAME + "/outs/binned_outputs/square_016um"
###

# area to be analyzed
# ## GCTB spatial G1, FFPE9
mask_large_x1, mask_large_x2, mask_large_y1, mask_large_y2 = 450, 1950, 250, 1750

# Species = "Human"
Species = "Mouse"

# List of target gene names
if Species == "Human":
    target_genes = [
        'TNFSF11' # Add more genes here if needed
    ]
    prefix_mt = 'MT-'
    file_nichenet = '../GCTB_bin2cell/data/tutorial/ligand_target_df.csv' # change as you like
else:
    target_genes = [
        'Col1a1' # Add more genes here if needed
    ]
    prefix_mt = 'mt-'
    file_nichenet = '../GCTB_bin2cell/data/tutorial/ligand_target_df_mouse.csv' # change as you like

sender_cell_type = ["Endothelial"]

### choose microenvironment clusters to be analyzed
cluster_label = ['12']

# Definition of neighborhood cells
neighbor_cell_numbers = 19

#role = 'sender'
role = 'receiver'
each_display_num = 5

# setting for filenames
label_image_filename = "he_labels_image.pdf"
h5ad_filename = SAMPLE_NAME + "_b2c.h5ad"
h5ad_full_filename = SAMPLE_NAME + "_2um.h5ad"
h5ad_predicted_full_filename = SAMPLE_NAME + "_nucleus_predicted.h5ad"
h5ad_sc_filtered_full_filename = SAMPLE_NAME + "_single_cell_filtered.h5ad"
h5ad_sc_microenvironment_full_filename = SAMPLE_NAME + "_single_cell_microenvironment.h5ad"
save_spatial_plot_path = os.path.join(save_path_for_today, "cropped_spatial_plot.svg")
save_svg_path = os.path.join(save_path_for_today, "spatial_salvage_labels.svg")
h5ad_save_path = os.path.join(save_path_for_today, h5ad_filename)
h5ad_full_save_path = os.path.join(save_path_for_today, h5ad_full_filename)
h5ad_predicted_full_save_path = os.path.join(save_path_for_today, h5ad_predicted_full_filename)
h5ad_sc_filtered_full_save_path = os.path.join(save_path_for_today, h5ad_sc_filtered_full_filename)
h5ad_microenvironment_full_save_path = os.path.join(save_path_for_today, h5ad_sc_microenvironment_full_filename)

os.chdir(path)
os.makedirs(save_path_for_today, exist_ok=True)

lt_df_raw = pd.read_csv(file_nichenet, index_col=0)
sp_adata_raw = sc.read_h5ad(h5ad_save_path)
sp_adata_microenvironment = sc.read_h5ad(h5ad_microenvironment_full_save_path)
sp_adata_microenvironment.obs['predicted_microenvironment'] = sp_adata_microenvironment.obs['predicted_microenvironment'].astype(str)

target_cell_type = sp_adata_microenvironment.obs['predicted_cell_type'].value_counts().idxmax()
# target_cell_type = 'Airway epithelial cells (CAPN8+, ELF3+)' # 'Tumor', 'GiantCell', etc., replace with your desired CellType
# target_cell_type = "Tumor"
# target_cell_type = "GiantCell"
# target_cell_type = "GC_proliferating"


## Ligand-receptor database: select Human or Mouse

In [ ]:
### Set proper csv file

# Human

### ligand-receptor data obtained from nichenet, download only once
# # 1. Google Drive link URL
# url = "https://drive.google.com/uc?export=download&id=1WpHzP_ticvL1T_aIufHI1ISx3VQMnUhd"
# # 2. File name
# output = "ligand_target_df.csv.zip"
# # 3. Download
# gdown.download(url, output, quiet=False)
# # 4. ZIP file extraction
# with zipfile.ZipFile(output, 'r') as zip_ref:
#     zip_ref.extractall("ligand_target_df")


# Mouse

### ligand-receptor data obtained from nichenet, download only once
# # 1. Google Drive link URL
# url = "https://drive.google.com/uc?export=download&id=1vD7MKPmOIpVGDO242bj9xHuBuvEzTrl3"
# # 2. File name
# output = "ligand_target_df.csv.zip"
# # 3. Download
# gdown.download(url, output, quiet=False)
# # 4. ZIP file extraction
# with zipfile.ZipFile(output, 'r') as zip_ref:
#     zip_ref.extractall("ligand_target_df")


In [ ]:
# DeepCOLOR-like analysis with NicheNet database
importlib.reload(huetracer.cci)
importlib.reload(huetracer.statistics)
importlib.reload(huetracer.plot)
importlib.reload(huetracer)

sp_adata, lt_df = huetracer.prepare_microenv_data(sp_adata_raw, sp_adata_microenvironment, lt_df_raw, min_frac=0.001, n_top_genes=500)

sp_adata = sp_adata.copy()
if scipy.sparse.issparse(sp_adata.X):
    sp_adata.X = sp_adata.X.toarray()
huetracer.add_zscore_layers(sp_adata, top_fraction=0.05)
top_diff_expr_genes = huetracer.make_top_values(sp_adata.layers["zscore_by_celltype"], axis=1, top_fraction=0.01)
expr_up_by_ligands = huetracer.make_top_values(top_diff_expr_genes @ lt_df, top_fraction=0.05).to_numpy()
ligands = lt_df.columns

edge_df, center_adata, exp_data = huetracer.construct_microenvironment_data(
    sp_adata=sp_adata,
    ligands=ligands,
    expr_up_by_ligands=expr_up_by_ligands,
    neighbor_cell_numbers=neighbor_cell_numbers
)

# Extract ligands with CCI values at least 1.25 times the mean
coexp_cc_df, bargraph_df = huetracer.calculate_enhanced_coexpression_coactivity(
    edge_df, center_adata, exp_data, expr_up_by_ligands, sp_adata, 
    role='receiver', up_rate=1.25
)

huetracer.comprehensive_interaction_analysis(coexp_cc_df, enhancement_threshold=1.25, spontaneous_threshold=0.1,
                                             inhibition_threshold=-0.05, min_responses_with_sender=10, min_responses_without_sender=10)

# # save CCI data as an excel file
filename = SAMPLE_NAME + "_environment_to_center__all_clusters__cell_type_cell_1_is_" + role + ".xlsx"
out_xlsx = os.path.join(save_path_for_today, filename)
coexp_cc_df.to_excel(out_xlsx)

# Extract ligands with CCI values at least 1.25 times the mean
coexp_cc_df_cluster, bargraph_df_cluster = huetracer.calculate_enhanced_coexpression_coactivity_cluster(edge_df, center_adata, exp_data, expr_up_by_ligands, sp_adata=sp_adata, cluster_label=cluster_label, role=role, up_rate = 1.25)
huetracer.comprehensive_interaction_analysis(coexp_cc_df_cluster, enhancement_threshold=1.25, spontaneous_threshold=0.1,
                                             inhibition_threshold=-0.05, min_responses_with_sender=10, min_responses_without_sender=10)

# save CCI data as an excel file
filename = f"{SAMPLE_NAME}_environment_to_center__selected_clusters__{cluster_label}__cell_type_cell_1_is_{role}.xlsx"
out_xlsx = os.path.join(save_path_for_today, filename)
coexp_cc_df_cluster.to_excel(out_xlsx)

# enhanced_significant: Effect of the presence or absence of the sending cell
# baseline_significant: Importance of cell-type-specific responses

# When baseline_significant = True:
#   For the receiving cell type, the signal from this ligand is physiologically important.
#   It triggers a response that exceeds the usual (baseline) state.
#   A cell-type-specific response mechanism is at work.

# When baseline_significant = False:
#   The observed response rate is within the normal variation range of that cell type.
#   The response may not be ligand-specific.
#   It could be noise or a non-specific reaction.

In [ ]:
# Evaluation of inter-cell-type proximity
results_df, fig = huetracer.analyze_cell_proximity(edge_df, save_path=save_path_for_today, n_permutations=100)

In [ ]:
importlib.reload(huetracer.plot)
huetracer.plot.interactive_cci_sankey(
    coexp_cc_df=coexp_cc_df,
    edge_df=edge_df,
    bargraph_df=bargraph_df,
    sp_adata=sp_adata,
    lib_id=lib_id,
    SAMPLE_NAME=SAMPLE_NAME,
    save_path_for_today=save_path_for_today,
    coexp_cc_df_cluster=coexp_cc_df_cluster,
    bargraph_df_cluster=bargraph_df_cluster
)